# Large Benchmark Sweeps With Progress\n\nThis notebook runs larger benchmark sweeps with heartbeat logs so long runs always show activity.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists() and (p / 'tools').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
print('repo:', REPO_ROOT)


In [ ]:
# Choose backend mode
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
if JAX_MODE not in {'cpu', 'gpu'}:
    raise ValueError(f'JAX_MODE must be cpu or gpu, got: {JAX_MODE}')
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
if JAX_DTYPE not in {'float64', 'float32'}:
    raise ValueError(f'JAX_DTYPE must be float64 or float32, got: {JAX_DTYPE}')
os.environ['JAX_PLATFORMS'] = 'cuda' if JAX_MODE == 'gpu' else 'cpu'

# Optional runtime tuning for shared GPU machines
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.85'

print('jax_dtype:', JAX_DTYPE)
expected = 'gpu' if JAX_MODE == 'gpu' else 'cpu'
!python tools/check_jax_runtime.py --expect-backend {expected} --quick-bench


In [ ]:
# Preview planned jobs without running
SAMPLES = os.getenv('EXAMPLE_SAMPLES', '800,1600')
SEEDS = os.getenv('EXAMPLE_SEEDS', '0,1')
CHUNK = os.getenv('EXAMPLE_CHUNK', '6')
!python tools/run_notebook_sweeps.py --dry-run --samples {SAMPLES} --seeds {SEEDS} --chunk-size {CHUNK} --jax-mode {JAX_MODE} --jax-dtype {JAX_DTYPE}


In [ ]:
# Run larger sweeps with status + heartbeat output
# Add --c-ref-dir /path/to/build to include C reference comparisons.
SAMPLES = os.getenv('EXAMPLE_SAMPLES', '800,1600')
SEEDS = os.getenv('EXAMPLE_SEEDS', '0,1')
CHUNK = os.getenv('EXAMPLE_CHUNK', '6')
!python tools/run_notebook_sweeps.py --samples {SAMPLES} --seeds {SEEDS} --chunk-size {CHUNK} --heartbeat 20 --jax-mode {JAX_MODE} --jax-dtype {JAX_DTYPE}
